In [44]:
import numpy as np
import time
from scipy.sparse.linalg import splu
from neuromodes.io import fetch_surf
from neuromodes.distances import geodesic_heat
from lapy import TriaMesh, Solver
from lapy.heat import diffusion
from lapy.diffgeo import tria_compute_geodesic_f

In [45]:
n = np.random.randint(50, 150)  # Randomly choose n between 10 and 100
pos = np.random.randint(0, n * n, size=(10,))  # Randomly choose a vertex index as source
print(f"Square with {n} vertices per side, total vertices: {n*n}")
print(f"Source vertex index: {pos}")

## generate square mesh with z coordinates all zero
# generate vertices
x, y = np.meshgrid(np.arange(n), np.arange(n), indexing='xy')
x = x.flatten().astype(np.float64)
y = y.flatten().astype(np.float64)
mask = (x != 0) & (x != n - 1) & (y != 0) & (y != n - 1)
x[mask] += np.random.uniform(low=-0.4, high=0.4, size=mask.sum())
y[mask] += np.random.uniform(low=-0.4, high=0.4, size=mask.sum())
vertices = np.stack([x/(n-1),y/(n-1),np.zeros_like(x)], axis=1).astype(np.float64)

# procedurally generate faces for square mesh
vi = np.arange(n * n).reshape(n, n)
top_left = vi[:-1, :-1].flatten()
top_right = top_left + 1
bottom_left = top_left + n
bottom_right = top_left + n + 1
tri1 = np.stack([top_left, top_right, bottom_left], axis=1)
tri2 = np.stack([top_right, bottom_right, bottom_left], axis=1)
faces = np.vstack([tri1, tri2])

# set up TriaMesh and some key features
T = TriaMesh(vertices, faces)
fem = Solver(T, lump=True)
nv = T.v.shape[0] 


Square with 104 vertices per side, total vertices: 10816
Source vertex index: [ 9036  7431 10008  4230  8311  4420  4113 10324  2926  5360]


In [46]:
## compare speed of two methods, for one vertex 
# lapy distances
tic = time.time()
dists_lapy = tria_compute_geodesic_f(T, diffusion(T, vids=pos[0]))
print(f'Time taken for LaPy to calculate 1 geodesic: {time.time()-tic:.5f} seconds')


# geodesic distances from new method
tic = time.time()
splu_stiffness = splu(fem.stiffness)
m = 1
t = m * T.avg_edge_length() ** 2
hmat = fem.mass + t * fem.stiffness
splu_heat = splu(hmat)

dists_splu = np.squeeze(geodesic_heat(T, (pos[0],), splu_heat, splu_stiffness))
print(f'Time taken for new method to calculate 1 geodesic: {time.time()-tic:.5f} seconds')

assert(np.allclose(dists_lapy, dists_splu))

Time taken for LaPy to calculate 1 geodesic: 0.23829 seconds
Time taken for new method to calculate 1 geodesic: 0.15372 seconds


In [47]:
## compare speed of two methods, for multiple vertices
# lapy distances
tic = time.time()
dists_lapy = np.zeros((nv, len(pos)))
for i, p in enumerate(pos): 
    dists_lapy[:,i] = tria_compute_geodesic_f(T, diffusion(T, vids=p))
print(f'Time taken for LaPy to calculate {len(pos)} geodesics: {time.time()-tic:.5f} seconds')


# geodesic distances from heat
tic = time.time()
splu_stiffness = splu(fem.stiffness)
m = 1
t = m * T.avg_edge_length() ** 2
hmat = fem.mass + t * fem.stiffness
splu_heat = splu(hmat)

dists_splu = np.squeeze(geodesic_heat(T, pos, splu_heat, splu_stiffness))
print(f'Time taken for new method to calculate {len(pos)} geodesics: {time.time()-tic:.5f} seconds')

assert(np.allclose(dists_lapy, dists_splu))


Time taken for LaPy to calculate 10 geodesics: 2.22112 seconds
Time taken for new method to calculate 10 geodesics: 0.50025 seconds


In [ ]:
mesh, _ = fetch_surf(density='4k', hemi='L', surf_type='sphere')
mesh.vertices = mesh.vertices / np.linalg.norm(mesh.vertices, axis=1, keepdims=True)  # Scale vertices to unit sphere

T = TriaMesh(mesh.vertices, mesh.faces)
fem = Solver(T, lump=True)

## compare geodesic distances from heat with euclidean distances
pos = np.random.randint(0, T.v.shape[0], size=(50,))  # Randomly choose a vertex index as source
print(f"Source vertex index: {pos}")


Source vertex index: [2325 3380 3094  788 2767 1787 2254 3591 3431  589 3716 2270 2804 2199
 1516 3106 1901 2346 2291 3378  384 3774  858 3123 2603 1202 3064 2718
 3521  317 3851 3148 2660  745 1172 1954 1089  464 2262  345  324 2677
 3288  440 1659 2488 3366 3102 1016 2804]


In [50]:
## compare speed of two methods, for one vertex 
# lapy distances
tic = time.time()
dists_lapy = tria_compute_geodesic_f(T, diffusion(T, vids=pos[0]))
print(f'Time taken for LaPy to calculate 1 geodesic: {time.time()-tic:.5f} seconds')


# geodesic distances from new method
tic = time.time()
splu_stiffness = splu(fem.stiffness)
m = 1
t = m * T.avg_edge_length() ** 2
hmat = fem.mass + t * fem.stiffness
splu_heat = splu(hmat)

dists_splu = np.squeeze(geodesic_heat(T, (pos[0],), splu_heat, splu_stiffness))
print(f'Time taken for new method to calculate 1 geodesic: {time.time()-tic:.5f} seconds')

assert(np.allclose(dists_lapy, dists_splu))

Time taken for LaPy to calculate 1 geodesic: 0.08470 seconds
Time taken for new method to calculate 1 geodesic: 0.07675 seconds


In [52]:
## compare speed of two methods, for multiple vertices
# lapy distances
tic = time.time()
dists_lapy = np.zeros((T.v.shape[0], len(pos)))
for i, p in enumerate(pos): 
    dists_lapy[:,i] = tria_compute_geodesic_f(T, diffusion(T, vids=p))
print(f'Time taken for LaPy to calculate {len(pos)} geodesics: {time.time()-tic:.5f} seconds')


# geodesic distances from heat
tic = time.time()
splu_stiffness = splu(fem.stiffness)
m = 1
t = m * T.avg_edge_length() ** 2
hmat = fem.mass + t * fem.stiffness
splu_heat = splu(hmat)

dists_splu = np.squeeze(geodesic_heat(T, pos, splu_heat, splu_stiffness))
print(f'Time taken for new method to calculate {len(pos)} geodesics: {time.time()-tic:.5f} seconds')

assert(np.allclose(dists_lapy, dists_splu))


Time taken for LaPy to calculate 50 geodesics: 3.99989 seconds
Time taken for new method to calculate 50 geodesics: 0.54000 seconds
